# Broadband SOM versus spec-z flow

Run `python bin/compare_som.py` from the repository root first. This comparison
uses only degraded broadband fluxes/errors and training spectra. No deep inputs,
COSMOS PDFs, or extra noise augmentation enter either method. Both models are
trained on 22,604 spectra and evaluated on the identical 5,083 held-out galaxies.
The SOM is a single-map cell-density estimator, **not full SOMPZ**.

Five seeds are fitted. SOM map size, log(1+z) smoothing bandwidth and shrinkage
are selected by validation negative log density using the same 4,096-object
subset as the first flow seed. Subsequent seeds keep that configuration fixed.
Cell PDFs use training spectra only; empty cells borrow the nearest populated
prototype. Each flow retains its original architecture and early stopping.

Plots use the mean of per-seed metrics. Magnitude shading uses 200 paired
resamples of a seed and the test sky cells within each bin. This captures seed
variation and evaluation-sample variation, but not retraining under a new
spectroscopic selection or full survey cosmic variance.

In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'setup.py').exists())
sys.path.insert(0, str(ROOT / 'src'))
from pzflow.utils import coverage

os.environ['PATH'] = '/Library/TeX/texbin:' + os.environ['PATH']
plt.rcParams.update(
    {
        'text.usetex': True,
        'font.family': 'serif',
        'font.size': 13,
        'axes.linewidth': 1.5,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.major.size': 5,
        'ytick.major.size': 5,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'legend.frameon': False,
    }
)
OUT = ROOT / 'results/som_comparison'
run = json.loads((OUT / 'run.json').read_text())
# CDF grids are used only below for n(z); load other arrays for all five seeds.
pred = []
for seed in run['seeds']:
    with np.load(OUT / f'predictions_{seed}.npz') as p:
        pred.append({k: p[k] for k in p.files if not k.endswith('_cdf')})
z, mag, cells = pred[0]['zspec'], pred[0]['imag'], pred[0]['cell']
for p in pred:
    assert np.array_equal(p['id'], pred[0]['id'])
    assert np.array_equal(p['flux'], pred[0]['flux'])
FIG = OUT / 'figures'
FIG.mkdir(exist_ok=True)
models = [('flow', 'Spec-z flow', 'C1'), ('som', 'Broadband SOM', 'C2')]
keys = [
    'bias',
    'sigma_nmad',
    'outlier_fraction',
    'coverage68',
    'coverage95',
    'crps',
    'mean_redshift_error',
]
summary = {
    key: {
        metric: {
            'mean': float(np.mean([r[key][metric] for r in run['runs']])),
            'seed_std': float(np.std([r[key][metric] for r in run['runs']], ddof=1)),
        }
        for metric in keys
    }
    for key, _, _ in models
}
(OUT / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Selected SOM:', run['selected'])
print(json.dumps(summary, indent=2))

In [ ]:
levels = np.linspace(0, 1, 101)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for key, label, color in models:
    curves = np.array([coverage(p[key + '_pit'], levels) for p in pred])
    axes[0].plot(levels, curves.mean(0), color=color, label=label)
    axes[0].fill_between(levels, curves.min(0), curves.max(0), color=color, alpha=0.25)
    hist = [np.histogram(p[key + '_pit'], bins=20, range=(0, 1), density=True)[0] for p in pred]
    axes[1].stairs(np.mean(hist, axis=0), np.linspace(0, 1, 21), color=color, label=label)
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].axhline(1, color='k', ls='--', lw=1)
axes[0].set(
    xlabel='Nominal central probability', ylabel='Empirical coverage', xlim=(0, 1), ylim=(0, 1)
)
axes[1].set(xlabel='PIT', ylabel='Probability density', xlim=(0, 1))
axes[0].legend(fontsize=11)
fig.tight_layout()
fig.savefig(FIG / 'coverage.png', dpi=180)

In [ ]:
def stat(p, key, idx):
    residual = (p[key + '_quantiles'][idx, 2] - z[idx]) / (1 + z[idx])
    return np.array(
        [
            1.4826 * np.median(abs(residual - np.median(residual))),
            coverage(p[key + '_pit'][idx], [0.68])[0],
        ]
    )


def resample_cells(idx, rng):
    unique = np.unique(cells[idx])
    groups = {c: idx[cells[idx] == c] for c in unique}
    return np.concatenate([groups[c] for c in rng.choice(unique, len(unique), replace=True)])


edges = [18.0, 22.5, 23.75, 24.5, 25.0]
rng = np.random.default_rng(20261121)
centers, counts = [], []
values = {key: [] for key, _, _ in models}
bounds = {key: [] for key, _, _ in models}
for lo, hi in zip(edges[:-1], edges[1:]):
    idx = np.flatnonzero((mag >= lo) & (mag < hi))
    centers.append(np.median(mag[idx]))
    counts.append(len(idx))
    draws = [(rng.integers(len(pred)), resample_cells(idx, rng)) for _ in range(200)]
    for key, _, _ in models:
        values[key].append(np.mean([stat(p, key, idx) for p in pred], axis=0))
        boot = np.array([stat(pred[s], key, ids) for s, ids in draws])
        bounds[key].append(np.quantile(boot, [0.025, 0.975], axis=0))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for key, label, color in models:
    v, b = np.array(values[key]), np.array(bounds[key])
    for j, ax in enumerate(axes):
        ax.plot(centers, v[:, j], 'o-', color=color, label=label)
        ax.fill_between(centers, b[:, 0, j], b[:, 1, j], color=color, alpha=0.5)
for ax in axes:
    for x, count in zip(centers, counts):
        ax.text(
            x, 1.02, f'$N={count}$', transform=ax.get_xaxis_transform(), ha='center', fontsize=10
        )
    ax.set_xlabel('HSC i aperture magnitude [AB]')
axes[0].set_ylabel(r'$\sigma_{\rm NMAD}$')
axes[1].set_ylabel('68 percent interval coverage')
axes[1].axhline(0.68, color='k', ls='--', lw=1)
axes[0].legend(fontsize=11)
fig.tight_layout()
fig.savefig(FIG / 'magnitude.png', dpi=180)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
edges = np.linspace(0, 3, 61)
axes[0].stairs(
    np.histogram(z, edges)[0] / (len(z) * np.diff(edges)), edges, color='k', label='Test spectra'
)
means = {key: [] for key, _, _ in models}
for seed in run['seeds']:
    with np.load(OUT / f'predictions_{seed}.npz') as p:
        grid = p['grid']
        for key, _, _ in models:
            means[key].append(p[key + '_cdf'].mean(0))
truth = np.mean(z[:, None] <= grid[None, :], axis=0)
for key, label, color in models:
    c = np.mean(means[key], axis=0)
    # Smooth only the displayed stacked density to reduce numerical differentiation noise.
    density = np.diff(np.interp(edges, grid, c)) / np.diff(edges)
    axes[0].stairs(density, edges, color=color, label=label)
    axes[1].plot(grid, c - truth, color=color, label=label)
axes[0].set(xlabel='Redshift', ylabel='Stacked redshift density', xlim=(0, 3))
axes[1].set(xlabel='Redshift', ylabel='Predicted minus empirical CDF', xlim=(0, 3))
axes[1].axhline(0, color='k', ls='--', lw=1)
axes[0].legend(fontsize=10)
fig.tight_layout()
fig.savefig(FIG / 'nz.png', dpi=180)